# Sara retrieve + rerank - refactored notebook

This notebook keeps the original experiment flow but imports reusable logic from `src/sara_retrieve_rerank`.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "sara_retrieve_rerank").exists():
            return path
    raise RuntimeError("Could not find project root. Open this notebook from inside sara_retrieve_rerank_project.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")


## Optional: install dependencies

Run the next cell only once in a fresh environment, or if imports fail.


In [ ]:
# Run this cell once per new kernel/environment if imports fail.
# In VS Code, prefer selecting .venv/bin/python as the notebook kernel first.
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)])


In [ ]:
from pathlib import Path

from sara_retrieve_rerank.config import (
    DEFAULT_CANDIDATES_PATH,
    DEFAULT_MATCHES_OUTPUT_PATH,
    DEFAULT_VACANCIES_PATH,
    EMBEDDING_MODEL,
    TOP_K,
)
from sara_retrieve_rerank.data import load_jsonl, write_jsonl
from sara_retrieve_rerank.documents import create_vacancy_documents
from sara_retrieve_rerank.evaluation import evaluate_retriever, missed_candidate_ids_at_k
from sara_retrieve_rerank.retrieval import retrieve_all_matches, retrieve_top_vacancies
from sara_retrieve_rerank.vector_store import create_vectorstore, index_documents
from sara_retrieve_rerank.visualization import plot_vacancies_stats

print("Imports ready")


## Load data

Put the two JSONL files in `../data/raw/` when running from this notebook.


In [ ]:
candidates = load_jsonl(Path('..') / DEFAULT_CANDIDATES_PATH)
vacancies = load_jsonl(Path('..') / DEFAULT_VACANCIES_PATH)

print(f"Loaded {len(candidates)} candidates")
print(f"Loaded {len(vacancies)} vacancies")


## Explore vacancy distributions

In [ ]:
plot_vacancies_stats(vacancies, 'specializations')
plot_vacancies_stats(vacancies, 'grades')
plot_vacancies_stats(vacancies, 'regions')


## Build vector index

In [ ]:
vacancy_docs = create_vacancy_documents(vacancies)
print(f"Created {len(vacancy_docs)} vacancy documents")

vectorstore = create_vectorstore(
    embedding_model=EMBEDDING_MODEL,
    persist_directory=None,  # in-memory, matching the original notebook behavior
    reset=True,
)
index_documents(vectorstore, vacancy_docs)
print("Vector store ready")


## Inspect one candidate

In [ ]:
candidate_index = 71
candidate = candidates[candidate_index]
print(candidate.get('text', ''))

matches = retrieve_top_vacancies(candidate, vectorstore, k=100)
for match in matches[:10]:
    print(
        match['rank'],
        match['vacancy_id'],
        round(match['cosine_similarity'], 4),
        match['title'],
        match['regions'],
    )


## Save all top-K matches

In [ ]:
all_matches = retrieve_all_matches(candidates, vectorstore, k=TOP_K)
output_path = Path('..') / DEFAULT_MATCHES_OUTPUT_PATH
write_jsonl(all_matches, output_path)
print(f"Saved {len(all_matches)} matches to {output_path}")


## Evaluate retrieval

In [ ]:
metrics = evaluate_retriever(candidates, vectorstore, ks=[1, 5, 10, 20, 50, 100])
for metric_name, value in metrics.items():
    print(metric_name, value)

missed_ids = missed_candidate_ids_at_k(candidates, vectorstore, k=100)
print(f"Missed candidates @100: {len(missed_ids)}")
print(missed_ids)
